<a href="https://www.kaggle.com/code/ps6nlb/267600821-analyze-sentiment?scriptVersionId=329395082" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install pythainlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 54.4 MB/s eta 0:00:00


# Import Library

In [2]:
import numpy as np 
import pandas as pd 
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import recall_score, precision_score, accuracy_score, classification_report

# สำรวจข้อมูล Wongnai-Food-Review-Rating-Prediction-Challenge

In [4]:
df = pd.read_csv('/kaggle/input/competitions/wongnai-food-review-rating-prediction-challenge/train_data.csv')
df

,review_body,star_rating
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,2
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,3
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,2
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,4
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,4
...,...,...
39995,รู้จักร้านนี้จากวงใน ร้านอยู่ในอาคารพาณิชย์ตรง...,3
39996,ร้านซูชิอาหารญี่ปุ่น อยู่ตรงสะพานลอย เกษตรประต...,3
39997,"""Cantina Wine Bar & Italian Kitchen"" ร้านเล็กๆ...",4
39998,ร้านเค้กน่ารักๆ ตรงชั้นล่างของห้างเซ็นทรัลลาดพ...,2


In [5]:
df.groupby('star_rating').count()

,review_body
star_rating,
0,415
1,1845
2,12171
3,18770
4,6799


# การจัดเตรียมข้อมูล (Data Preparation)

แปลงตัวแปรเป้าหมาย (Label) จาก star_rating เป็นค่าประเมินความคิดเห็นตามโจทย์ <br> Start_Rating 0-1 แปลงเป็นความคิดเห็นเชิงลบ (neg : 0) <br> Start_Rating 2 แปลงเป็นความคิดเห็นทั่วไป (gen : 1) <br>Start_Rating 3-4 แปลงเป็นความคิดเห็นเชิงบวก (pos : 2)

In [6]:
df_neg = df[(df['star_rating'] == 0) | (df['star_rating'] == 1) ]
df_neg

,review_body,star_rating
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,1
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,1
101,ไม่รู้จะใจร้ายไปรึเปล่า แต่ไม่อร่อยอ่ะค่ะ\n\n1...,0
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,1
111,วนเวียนอยู่แถวซอยอารีย์ หาร้านส้มตำทานกันเถอะๆ...,1
...,...,...
39869,เรียกว่าเป็นร้านที่โดยส่วนตัวแล้วค่อนข้างเซอร์...,1
39896,#Bangkok N°329/T982/P11605+(6)\n\nอาทิตย์​ก่อน...,1
39922,การปรับปรุงเมนูและวัตถุดิบจากครั้งล่าสุด ทำให้...,1
39938,สำหรับเรา เจียงเป็นร้านก๋วยเตี๋ยวร้านดังที่ไม่...,1


In [7]:
df_neg.loc[:, ['star_rating']] = 0
df_neg

,review_body,star_rating
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,0
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,0
101,ไม่รู้จะใจร้ายไปรึเปล่า แต่ไม่อร่อยอ่ะค่ะ\n\n1...,0
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0
111,วนเวียนอยู่แถวซอยอารีย์ หาร้านส้มตำทานกันเถอะๆ...,0
...,...,...
39869,เรียกว่าเป็นร้านที่โดยส่วนตัวแล้วค่อนข้างเซอร์...,0
39896,#Bangkok N°329/T982/P11605+(6)\n\nอาทิตย์​ก่อน...,0
39922,การปรับปรุงเมนูและวัตถุดิบจากครั้งล่าสุด ทำให้...,0
39938,สำหรับเรา เจียงเป็นร้านก๋วยเตี๋ยวร้านดังที่ไม่...,0


In [8]:
df_gen = df[df['star_rating'] == 2]
df_gen

,review_body,star_rating
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,2
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,2
7,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,2
22,ระหว่างทางไปเขาค้อ ผมได้คำแนะนำจากเด็กปั๊มให้ม...,2
31,ร้านอาหารเรื่อนปั้นหยา ที่จอดรถกว้างขวางสะดวกส...,2
...,...,...
39989,ร้านข้าวมันไก่ในซอยวัดหนองขาม มาแวะซื้อใส่กล่อ...,2
39990,ร้านอยู่กลางเมกะบางนา ตรงประชาสัมพันธ์ ใต้บันไ...,2
39993,นั่งกิน Bar B Q กันอยู่ มองไปทาง food court เห...,2
39998,ร้านเค้กน่ารักๆ ตรงชั้นล่างของห้างเซ็นทรัลลาดพ...,2


In [9]:
df_gen.loc[:, ['star_rating']] = 1
df_gen

,review_body,star_rating
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,1
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,1
7,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,1
22,ระหว่างทางไปเขาค้อ ผมได้คำแนะนำจากเด็กปั๊มให้ม...,1
31,ร้านอาหารเรื่อนปั้นหยา ที่จอดรถกว้างขวางสะดวกส...,1
...,...,...
39989,ร้านข้าวมันไก่ในซอยวัดหนองขาม มาแวะซื้อใส่กล่อ...,1
39990,ร้านอยู่กลางเมกะบางนา ตรงประชาสัมพันธ์ ใต้บันไ...,1
39993,นั่งกิน Bar B Q กันอยู่ มองไปทาง food court เห...,1
39998,ร้านเค้กน่ารักๆ ตรงชั้นล่างของห้างเซ็นทรัลลาดพ...,1


In [10]:
df_pos = df[(df['star_rating'] == 3) | (df['star_rating'] == 4) ]
df_pos.loc[:, ['star_rating']] = 2
df_pos

,review_body,star_rating
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,2
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,2
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,2
5,ร้านข้าวต้มกระดูกหมู ปากซอยพัฒนาการ 57 เป็นอีก...,2
6,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...,2
...,...,...
39992,ร้านที่ชื่อแปลก ตกแต่งสไตล์loft ได้อย่างลงตัว ...,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2
39995,รู้จักร้านนี้จากวงใน ร้านอยู่ในอาคารพาณิชย์ตรง...,2
39996,ร้านซูชิอาหารญี่ปุ่น อยู่ตรงสะพานลอย เกษตรประต...,2


In [11]:
df1 = pd.concat([df_neg, df_gen, df_pos])
df1

,review_body,star_rating
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,0
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,0
101,ไม่รู้จะใจร้ายไปรึเปล่า แต่ไม่อร่อยอ่ะค่ะ\n\n1...,0
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0
111,วนเวียนอยู่แถวซอยอารีย์ หาร้านส้มตำทานกันเถอะๆ...,0
...,...,...
39992,ร้านที่ชื่อแปลก ตกแต่งสไตล์loft ได้อย่างลงตัว ...,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2
39995,รู้จักร้านนี้จากวงใน ร้านอยู่ในอาคารพาณิชย์ตรง...,2
39996,ร้านซูชิอาหารญี่ปุ่น อยู่ตรงสะพานลอย เกษตรประต...,2


แบ่งข้อมูล Train Set/Test Set ในอัตราส่วน 80:20 โดยวิธีการสุ่ม (Sample)

In [12]:
df_train = df1.sample(32000)
df_train

,review_body,star_rating
21236,ไม่แน่ใจว่าพนักงานนับช้อนชา (น้ำตาล) ผิดสูตร ห...,0
15213,ได้ทานขนมปัง Danish กรอบ หอม อร่อยมากค่ะ เป็นข...,2
19766,ร้านซาลาเปา ติ่มซำ เฟรนไชส์ ชื่อดัง วราภรณ์ มี...,2
25729,ตะลอนเดินเล่นพื้นที่อ่าวนาง เห็นมีร้านขายโรตีเ...,1
34776,วันนี้โอกาสนัดสังสรรค์กับเดอะแกงค์หาที่รวมตัวส...,2
...,...,...
12952,ราคาค่อนข้างแรงมาก เมื่อเทียบกับรสชาติ ในความร...,2
4468,นึกถึงก๋วยเตี๋ยวเจ้าดังย่านสยาม คงนึกถึงร้านรส...,1
27071,สั่งมาเมนูเดียว คือ ซี่โครงหมูย่าง ราคาแพง เลย...,0
31342,ร้านหวานละมุนร้านขนมไทยเจ้าดังในเมืองเชียงใหม่...,1


In [13]:
df_test = df1.drop(df_train.index)
df_test

,review_body,star_rating
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,0
193,ก๋วยเตี๋ยวไก่อร่อยดีแต่ถ้วยเล็กไปหน่อย ส้ม\nตำ...,0
243,วันนี้ได้แวะไปทำธุระแถวบิ๊กซี สะพานใหม่ครับ..ก...,0
340,กินแค่หม้อไฟเชอรอยกับเนื้อตุ๋น หม้อไฟเปนโถเท่า...,0
415,ร้านเป็ดพะโล้นายสุนเจริญนคร มีอาหารหลากหลาย ทั...,0
...,...,...
39971,รู้สึกว้าววกับเมนูนี้มากเลยยค่าปป ผมรู้สึกมันเ...,2
39973,พนักงานบริการดี อาหารคุ้มกับราคาค่ะ ร้านใหญ่ ภ...,2
39981,ขนมจีนน้ำยาปู หาร้านทำอร่อยยากอยู่นะ เคยกินบาง...,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2


แยกคำที่มีในพจนานุกรมภาษาไทยลงใน List โดยใช้ไลบรารี่ pythainlp

In [14]:
def token_thai(text):
    tokens = word_tokenize(text, engine='newmm')
    return [word for word in tokens if word.strip() != '']

In [15]:
token_thai('ทดสอบแยกคำในภาษาไทย')

['ทดสอบ', 'แยก', 'คำ', 'ใน', 'ภาษาไทย']

นำ Data Set ไปวิเคราะห์ตาราง TF-IDF เพื่อหาคำเพื่อปราฏบ่อยที่สุดในเอกสาร

In [16]:
vectorizer = TfidfVectorizer(tokenizer=token_thai, token_pattern=None, max_features=10000, min_df= 5, max_df=0.9)
x_train = vectorizer.fit_transform(df_train['review_body'])

In [17]:
y_train = df_train['star_rating']
y_train

21236    0
15213    2
19766    2
25729    1
34776    2
        ..
12952    2
4468     1
27071    0
31342    1
34888    2
Name: star_rating, Length: 32000, dtype: int64

In [18]:
x_test = vectorizer.transform(df_test['review_body'])
x_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 678677 stored elements and shape (8000, 10000)>

In [19]:
y_test = df_test['star_rating']
y_test

72       0
193      0
243      0
340      0
415      0
        ..
39971    2
39973    2
39981    2
39991    2
39996    2
Name: star_rating, Length: 8000, dtype: int64

# การประมวลผลการพยากรณ์โดยใช้ Machine Learning / การประเมินโมเดล Classification

In [20]:
model_nai = MultinomialNB()
model_nai.fit(x_train, y_train)
y_pred = model_nai.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       449
           1       0.61      0.05      0.09      2440
           2       0.65      0.99      0.79      5111

    accuracy                           0.65      8000
   macro avg       0.42      0.35      0.29      8000
weighted avg       0.60      0.65      0.53      8000



In [21]:
model_svc = LinearSVC()
model_svc.fit(x_train, y_train)
y_pred = model_svc.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.61      0.32      0.42       449
           1       0.54      0.44      0.48      2440
           2       0.76      0.86      0.81      5111

    accuracy                           0.70      8000
   macro avg       0.64      0.54      0.57      8000
weighted avg       0.68      0.70      0.69      8000



In [22]:
model_dt =  DecisionTreeClassifier()
model_dt.fit(x_train, y_train)
y_pred = model_dt.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.17      0.16      0.17       449
           1       0.39      0.38      0.39      2440
           2       0.71      0.72      0.71      5111

    accuracy                           0.58      8000
   macro avg       0.42      0.42      0.42      8000
weighted avg       0.58      0.58      0.58      8000



เลือกใช้ Model ที่มีค่าผลการประเมินที่ดีที่สุดนั้นคือโมเดล LogisticRegression Classification

In [23]:
model_lr = LogisticRegression()
model_lr.fit(x_train, y_train)
y_pred = model_lr.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.72      0.29      0.41       449
           1       0.57      0.43      0.49      2440
           2       0.76      0.89      0.82      5111

    accuracy                           0.71      8000
   macro avg       0.68      0.53      0.57      8000
weighted avg       0.70      0.71      0.69      8000



In [24]:
y_pred

array([2, 1, 1, ..., 2, 2, 2], shape=(8000,))

หา Feature Impotant จาก Model ที่ดีที่สุด เพื่อหาคำที่มีผลต่อการทำนายของโมเดลมากที่สุด โดยแบ่งเป็น 3 คลาส ได้แก่ ความคิดเห็นเชิงลบ (neg), ความคิดเห็นทั่วไป (gen), ความคิดเห็นเชิงบวก (pos)

In [25]:
df_feature_neg = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[0])})
df_feature_neg.sort_values(by = 'Coef', ascending=False)[0:9]

,Feature,Coef
9848,ไม่,5.381274
9850,ไม่ค่อย,3.943750
2957,จืด,3.209024
9278,แย่,2.976575
5159,พอ,2.775702
9281,แย่มาก,2.768975
3402,ดีกว่า,2.481221
2392,คง,2.385391
9011,แข็ง,2.349542


In [26]:
df_feature_gen = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[1])})
df_feature_gen.sort_values(by = 'Coef', ascending=False)[0:9]

,Feature,Coef
9680,ใช้ได้,3.032531
1897,กลางๆ,2.857723
5181,พอใช้ได้,2.395614
4361,นิด,1.783146
3838,ถึงกับ,1.738569
9996,⭐️⭐️⭐️,1.717325
3399,ดี,1.711394
7825,เคเอฟซี,1.589774
8139,เท่าที่ควร,1.529452


In [27]:
df_feature_pos = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[2])})
df_feature_pos.sort_values(by = 'Coef', ascending=False)[0:9]

,Feature,Coef
7350,อร่อย,5.453091
3046,ชอบ,3.426045
3407,ดีมาก,3.089285
3405,ดีงาม,3.086847
8029,เด็ด,2.982443
2039,กำลังดี,2.749755
5328,ฟิน,2.675031
5146,พลาด,2.622495
1879,กลมกล่อม,2.593131


# การทดสอบชุดข้อมูล Test Set

นำข้อมูลจากการทำนายโมเดลมาเปรียบเทียบกับข้อมูลจริงในชุดข้อมูล TestSet

In [28]:
df_ex = df_test.copy()
df_ex['predict'] = y_pred
df_ex

,review_body,star_rating,predict
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,0,2
193,ก๋วยเตี๋ยวไก่อร่อยดีแต่ถ้วยเล็กไปหน่อย ส้ม\nตำ...,0,1
243,วันนี้ได้แวะไปทำธุระแถวบิ๊กซี สะพานใหม่ครับ..ก...,0,1
340,กินแค่หม้อไฟเชอรอยกับเนื้อตุ๋น หม้อไฟเปนโถเท่า...,0,1
415,ร้านเป็ดพะโล้นายสุนเจริญนคร มีอาหารหลากหลาย ทั...,0,0
...,...,...,...
39971,รู้สึกว้าววกับเมนูนี้มากเลยยค่าปป ผมรู้สึกมันเ...,2,2
39973,พนักงานบริการดี อาหารคุ้มกับราคาค่ะ ร้านใหญ่ ภ...,2,2
39981,ขนมจีนน้ำยาปู หาร้านทำอร่อยยากอยู่นะ เคยกินบาง...,2,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2,2


In [29]:
df_ex.to_csv('result1.csv', encoding='utf-8-sig')

กรองข้อมูลเฉพาะโมเดลที่ทำนายถูก

In [30]:
df_ex_true = df_ex[df_ex['star_rating'] == df_ex['predict']]
df_ex_true

,review_body,star_rating,predict
415,ร้านเป็ดพะโล้นายสุนเจริญนคร มีอาหารหลากหลาย ทั...,0,0
442,กราบขออภัยสำหรับ...คำพูด แต่ผมมีความรู้สึกอย่า...,0,0
687,เคยทานเมื่อปีที่แล้วละเลิกทานไปเลยค่ะ เป็นร้าน...,0,0
896,มาครั้งแรกจะไปตัดสินใจอะไรได้น้อ ผมเดินจากลานจ...,0,0
962,ร้านนี้เป็นร้านที่เราไปกินมานานมากแล้วเหมือนกั...,0,0
...,...,...,...
39971,รู้สึกว้าววกับเมนูนี้มากเลยยค่าปป ผมรู้สึกมันเ...,2,2
39973,พนักงานบริการดี อาหารคุ้มกับราคาค่ะ ร้านใหญ่ ภ...,2,2
39981,ขนมจีนน้ำยาปู หาร้านทำอร่อยยากอยู่นะ เคยกินบาง...,2,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2,2


In [31]:
df_ex_true[df_ex_true['predict'] == 2]

,review_body,star_rating,predict
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,2,2
14,"""ประจักษ์เป็ดย่าง"" \n\nไม่ใช่ร้านของพันตรีประจ...",2,2
27,"กินจนติดมาก เป็นร้านที่กินแล้ว ""สนุกลิ้น"" มากจ...",2,2
28,เพิ่งจะเปิดได้ไม่กี่ปีแต่ต้องยอมรับว่าเค้าประส...,2,2
46,ตอนนี้ทิง-ทิงคาราโอเกะ ได้ย้ายมาอยู่ อ่อนนุช29...,2,2
...,...,...,...
39971,รู้สึกว้าววกับเมนูนี้มากเลยยค่าปป ผมรู้สึกมันเ...,2,2
39973,พนักงานบริการดี อาหารคุ้มกับราคาค่ะ ร้านใหญ่ ภ...,2,2
39981,ขนมจีนน้ำยาปู หาร้านทำอร่อยยากอยู่นะ เคยกินบาง...,2,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2,2


กรองข้อมูลเฉพาะโมเดลที่ทำนายผิด

In [32]:
df_ex_fault = df_ex.drop(df_ex_true.index)
df_ex_fault

,review_body,star_rating,predict
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,0,2
193,ก๋วยเตี๋ยวไก่อร่อยดีแต่ถ้วยเล็กไปหน่อย ส้ม\nตำ...,0,1
243,วันนี้ได้แวะไปทำธุระแถวบิ๊กซี สะพานใหม่ครับ..ก...,0,1
340,กินแค่หม้อไฟเชอรอยกับเนื้อตุ๋น หม้อไฟเปนโถเท่า...,0,1
562,ร้านนี้น่าจะเรียกได้ว่าเป็นร้านที่มีอาหารจานเด...,0,1
...,...,...,...
39748,ตอน: ข้าวหมูแดงไม่มีชื่อร้าน แต่มีเปิบพิสดาร\n...,2,1
39795,ร้านสาขาจากกรุงเทพ.. ตกแต่งเหลืองดำ สดใส เมนูส...,2,1
39843,เวลาแวะที่ Gourmet ทีไรก็ต้องแวะซื้อน้ำเต้าหู้...,2,1
39867,มาทำงานในพื้นที่เทศบาลท่าเรือ พรรคพวกแนะนำให้ม...,2,1


In [33]:
df_ex_fault[df_ex_fault['predict'] == 1]

,review_body,star_rating,predict
193,ก๋วยเตี๋ยวไก่อร่อยดีแต่ถ้วยเล็กไปหน่อย ส้ม\nตำ...,0,1
243,วันนี้ได้แวะไปทำธุระแถวบิ๊กซี สะพานใหม่ครับ..ก...,0,1
340,กินแค่หม้อไฟเชอรอยกับเนื้อตุ๋น หม้อไฟเปนโถเท่า...,0,1
562,ร้านนี้น่าจะเรียกได้ว่าเป็นร้านที่มีอาหารจานเด...,0,1
662,ข้อดี อาหารสด น้ำจิ้มใช้ได้ วิวติดแม่น้ำ ร้านค...,0,1
...,...,...,...
39748,ตอน: ข้าวหมูแดงไม่มีชื่อร้าน แต่มีเปิบพิสดาร\n...,2,1
39795,ร้านสาขาจากกรุงเทพ.. ตกแต่งเหลืองดำ สดใส เมนูส...,2,1
39843,เวลาแวะที่ Gourmet ทีไรก็ต้องแวะซื้อน้ำเต้าหู้...,2,1
39867,มาทำงานในพื้นที่เทศบาลท่าเรือ พรรคพวกแนะนำให้ม...,2,1


# การประยุกต์นำเอาไปใช้งาน โดยการสร้างฟังก์ชัน analyze_sentiment (sentence) และทดสอบข้อความอื่นๆ ที่ไม่มีใน Dataset

In [34]:
def analyze_sentiment (sentence) :
    x_test = vectorizer.transform(pd.Series(sentence))
    y_pred = model_lr.predict(x_test)
    
    if y_pred.item() == 0:
        y_comment = 'ลบ'
    elif y_pred.item() == 1:
        y_comment = 'ทั่วไป'
    else:
        y_comment = 'บวก' 
        
    print(f'ความคิดเห็นเชิง : {y_comment}')

ทดสอบประโยคนอกเหนือจากใน Dataset

In [35]:
# ความคาดหวังความคิดเห็นเชิงบวก
sentence = 'เนื้อย่างพรีเมียมมาก นุ่มละลายในปาก พนักงานดูแลใส่ใจดี เปลี่ยนตะแกรงให้ตลอด คุ้มค่าสมราคาครับ แนะนำเลยไม่ผิดหวังแน่นอน'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : บวก


In [36]:
# ความคาดหวังความคิดเห็นเชิงลบ
sentence = 'ร้านตกแต่งสวยดี ถ่ายรูปมุมไหนก็สวย แต่อาหารรสชาติกลางๆ ไม่ได้โดดเด่นอะไรเมื่อเทียบกับร้านอื่น ราคาแอบแรงไปนิดนึง'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : ทั่วไป


In [37]:
# ความคาดหวังความคิดเห็นเชิงบวก
sentence = 'เห็นคนรีวิวเยอะว่าอร่อยที่สุดในย่านนี้ เลยตั้งความหวังไว้สูงมาก แต่พอกินจริงๆ คือธรรมดามาก ไม่ได้ว้าวอย่างที่คิดเลย แอบผิดหวัง'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : ลบ
